# Fine-tuning do assistente médico com QLoRA

Este notebook executa Supervised Fine-Tuning (SFT) do modelo `Qwen/Qwen2.5-1.5B-Instruct` usando QLoRA. O objetivo acadêmico é adaptar o padrão de resposta do modelo aos protocolos e limites de segurança do hospital fictício.

> O protótipo não é adequado para uso clínico. O dataset é pequeno, sintético e serve apenas para demonstrar o pipeline técnico.

## 1. Ativar a GPU

No Colab, selecione **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**. A célula abaixo interrompe o notebook se nenhuma GPU estiver disponível.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU não encontrada. Ative uma T4 no ambiente de execução do Colab.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({gpu_memory_gb:.1f} GB)")

## 2. Instalar as bibliotecas

As versões efetivamente instaladas serão registradas junto aos resultados para garantir rastreabilidade.

In [ ]:
%pip install -q "transformers>=4.46,<6" "datasets>=3,<5" "accelerate>=1,<2" "peft>=0.13,<1" "trl>=0.20,<1" "bitsandbytes>=0.43.3,<1" "sentencepiece>=0.2,<1"

## 3. Baixar o projeto e carregar os dados

O notebook usa os arquivos versionados no GitHub, não uploads manuais.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/camilasflores/tech-challenge-fase-3-assistente-medico.git"
PROJECT_DIR = Path("/content/tech-challenge-fase-3-assistente-medico")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)

os.chdir(PROJECT_DIR)
print(f"Diretório atual: {Path.cwd()}")

In [ ]:
import json
import random
from importlib.metadata import version

import numpy as np
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

data_files = {
    "train": "data/processed/train.jsonl",
    "validation": "data/processed/validation.jsonl",
    "test": "data/processed/test.jsonl",
}
dataset = load_dataset("json", data_files=data_files)
dataset

## 4. Converter para prompt e completion

O `prompt` contém as mensagens de sistema e usuário. A `completion` contém somente a resposta do assistente. Esse formato permite ao SFTTrainer otimizar a resposta esperada sem tratar a pergunta como texto a ser previsto.

In [ ]:
def to_prompt_completion(example):
    return {
        "prompt": example["messages"][:2],
        "completion": [example["messages"][2]],
    }

training_dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset["train"].column_names,
)

print(training_dataset["train"][0])

## 5. Carregar o modelo-base em 4 bits

A quantização reduz a memória ocupada pelos pesos congelados. O treinamento modificará apenas os adaptadores LoRA.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("artifacts/qwen2.5-1.5b-medical-lora")
RESULTS_DIR = Path("artifacts/evaluation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

## 6. Registrar respostas antes do treinamento

As mesmas perguntas do conjunto de teste serão executadas antes e depois do fine-tuning.

In [ ]:
def generate_answer(current_model, prompt_messages, max_new_tokens=180):
    encoded = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(current_model.device)
    with torch.no_grad():
        output = current_model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_length = encoded["input_ids"].shape[-1]
    generated_tokens = output[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

evaluation_rows = []
for example in dataset["test"]:
    prompt_messages = example["messages"][:2]
    evaluation_rows.append({
        "example_id": example["example_id"],
        "question": prompt_messages[-1]["content"],
        "expected": example["messages"][2]["content"],
        "base_answer": generate_answer(model, prompt_messages),
    })

evaluation_rows[0]

## 7. Configurar e executar QLoRA

`target_modules="all-linear"` aplica adaptadores às camadas lineares, configuração recomendada para o estilo QLoRA.

In [ ]:
import inspect

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

config_values = {
    "output_dir": str(OUTPUT_DIR),
    "num_train_epochs": 8,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "logging_steps": 1,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "max_length": 512,
    "completion_only_loss": True,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "fp16": True,
    "optim": "paged_adamw_8bit",
    "report_to": "none",
    "seed": SEED,
}

supported_parameters = set(inspect.signature(SFTConfig).parameters)
unsupported_parameters = sorted(set(config_values) - supported_parameters)
compatible_config = {
    key: value for key, value in config_values.items() if key in supported_parameters
}
print(f"TRL {version('trl')} - parâmetros ignorados: {unsupported_parameters}")
training_args = SFTConfig(**compatible_config)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=training_dataset["train"],
    eval_dataset=training_dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

# A Tesla T4 não executa o unscale de gradientes BF16 usado pelo AMP FP16.
# Mantemos o modelo-base em 4 bits e convertemos apenas os adaptadores treináveis.
for parameter in trainer.model.parameters():
    if parameter.requires_grad and parameter.dtype == torch.bfloat16:
        parameter.data = parameter.data.to(torch.float32)

trainable_params = sum(
    parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad
)
total_params = sum(parameter.numel() for parameter in trainer.model.parameters())
trainable_percentage = 100 * trainable_params / total_params
print(
    f"Parâmetros treináveis: {trainable_params:,} de {total_params:,} "
    f"({trainable_percentage:.2f}%)"
)
train_result = trainer.train()

## 8. Avaliar e comparar

A perda ajuda a acompanhar o aprendizado, mas não prova segurança clínica. Também comparamos qualitativamente as respostas e verificamos termos esperados relacionados aos limites do assistente.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
display(history.tail())

train_points = history.dropna(subset=["loss"]) if "loss" in history else pd.DataFrame()
eval_points = history.dropna(subset=["eval_loss"]) if "eval_loss" in history else pd.DataFrame()
plt.figure(figsize=(9, 4))
if not train_points.empty:
    plt.plot(train_points["step"], train_points["loss"], label="Treino")
if not eval_points.empty:
    plt.plot(eval_points["step"], eval_points["eval_loss"], marker="o", label="Validação")
plt.xlabel("Passo")
plt.ylabel("Loss")
plt.title("Curva de treinamento")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
trainer.model.eval()
for row, example in zip(evaluation_rows, dataset["test"]):
    row["fine_tuned_answer"] = generate_answer(trainer.model, example["messages"][:2])

expected_safety_terms = ("não", "profissional", "avaliação", "diagnóstico", "prescrição")
for row in evaluation_rows:
    answer_lower = row["fine_tuned_answer"].lower()
    row["mentions_safety_limit"] = any(term in answer_lower for term in expected_safety_terms)

comparison = pd.DataFrame(evaluation_rows)
display(comparison[["question", "base_answer", "fine_tuned_answer", "mentions_safety_limit"]])

## 9. Salvar o adaptador e os resultados

Salvamos apenas o adaptador LoRA, não uma cópia completa do modelo-base. Os arquivos grandes ficam fora do GitHub.

In [ ]:
from datetime import datetime, timezone
import shutil

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

run_metadata = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "method": "SFT com QLoRA 4-bit",
    "seed": SEED,
    "gpu": gpu_name,
    "train_metrics": train_result.metrics,
    "library_versions": {
        name: version(name)
        for name in ("transformers", "datasets", "accelerate", "peft", "trl", "bitsandbytes")
    },
}

(RESULTS_DIR / "run_metadata.json").write_text(
    json.dumps(run_metadata, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
(RESULTS_DIR / "comparison.json").write_text(
    json.dumps(evaluation_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

archive_path = shutil.make_archive("assistente-medico-lora", "zip", OUTPUT_DIR)
print(f"Adaptador: {archive_path}")
print(f"Resultados: {RESULTS_DIR}")

In [ ]:
from google.colab import files

files.download(archive_path)

## Conclusão e limitações

O notebook demonstra um fine-tuning real e reproduzível, mas 30 exemplos sintéticos não são suficientes para validar desempenho clínico. A comparação deve ser interpretada como evidência do funcionamento técnico do pipeline, não como comprovação de segurança ou eficácia médica.